In [1]:
import numpy as np

from qmodem import CMAPSSAnalyst

np.random.seed(42)

In [2]:
analyst = CMAPSSAnalyst(relative_test_size=0.2)
metrics_cmapss = analyst.compute_prognostic_metrics()

In [3]:
metrics_cmapss

,sensor_name,monotonicity,prognosability,trendability,fitness
0,sensor_11,0.811471,0.868111,0.019371,0.566318
1,sensor_12,-0.783066,0.843485,0.020273,0.548941
2,sensor_4,0.771310,0.871807,0.000436,0.547851
3,sensor_7,-0.751420,0.838661,0.001006,0.530362
4,sensor_15,0.710282,0.840112,0.000562,0.516985
5,sensor_20,-0.705609,0.827961,0.000794,0.511455
6,sensor_21,-0.699806,0.808456,0.000245,0.502836
7,sensor_2,0.661837,0.814902,0.000231,0.492323
8,sensor_17,0.644804,0.783602,0.000216,0.476208
9,sensor_3,0.631791,0.763712,0.000569,0.465357


Select the 9 sensors with the highest fitness.

In [4]:
analyst.train_df.head()

,unit_id,time_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
10980,56,1,-0.0009,-0.0005,100.0,642.57,1591.56,1405.96,553.34,2388.12,9055.81,47.38,521.93,2388.10,8133.85,8.4444,391,38.75,23.3174,274
10981,56,2,0.0012,-0.0004,100.0,642.75,1586.44,1415.44,552.68,2388.10,9059.10,47.72,521.18,2388.13,8136.92,8.4412,395,38.81,23.2391,273
10982,56,3,0.0012,-0.0004,100.0,642.47,1584.96,1410.81,552.90,2388.12,9057.99,47.42,520.82,2388.08,8133.11,8.4461,394,38.82,23.3340,272
10983,56,4,0.0026,0.0005,100.0,642.52,1587.64,1403.70,553.52,2388.13,9054.91,47.48,521.70,2388.12,8136.86,8.4357,394,38.89,23.2844,271
10984,56,5,0.0034,-0.0002,100.0,642.51,1587.80,1411.17,553.60,2388.13,9045.30,47.49,522.06,2388.18,8132.53,8.4411,394,38.79,23.3204,270


In [5]:
sensors_selected = metrics_cmapss.head(9)["sensor_name"].tolist()
train_df, test_df = analyst.train_df, analyst.test_df

# filter the dataframes to only include the selected sensors (exclude also the operati)
non_sensor_columns = [
    col
    for col in analyst.column_names
    if not (col.startswith("sensor") or col.startswith("op_setting"))
]
columns_selected = non_sensor_columns + sensors_selected
train_df = train_df[columns_selected]
test_df = test_df[columns_selected]

In [6]:
train_df.head()

,unit_id,time_cycles,RUL,sensor_11,sensor_12,sensor_4,sensor_7,sensor_15,sensor_20,sensor_21,sensor_2,sensor_17
10980,56,1,274,47.38,521.93,1405.96,553.34,8.4444,38.75,23.3174,642.57,391
10981,56,2,273,47.72,521.18,1415.44,552.68,8.4412,38.81,23.2391,642.75,395
10982,56,3,272,47.42,520.82,1410.81,552.90,8.4461,38.82,23.3340,642.47,394
10983,56,4,271,47.48,521.70,1403.70,553.52,8.4357,38.89,23.2844,642.52,394
10984,56,5,270,47.49,522.06,1411.17,553.60,8.4411,38.79,23.3204,642.51,394


Now, we can use the `CMAPSSDataSource` class to prepare the feature/label arrays for the train and test dataframes.
We use a standard scaler to bring the features to zero mean and variance = 1, in order to prevent the different scales of the features to condition the training.
We also define a time-window the histories, with a window length large enough to reveal degradation and small enough to prevent overfitting and have enough training samples.

In [7]:
from sklearn.preprocessing import StandardScaler

from qmodem.data import CMAPSSDataSource

scaler = StandardScaler()
window_size = 30

# Note, scaling happens at init time, so when the same scaler is passed to the test datasource, it will use the
# statistics computed on the training data to scale the test data.
ds_train = CMAPSSDataSource(
    train_df, train_or_test="train", scaler=scaler, window_size=window_size
)
ds_test = CMAPSSDataSource(
    test_df, train_or_test="test", scaler=scaler, window_size=window_size
)

In [8]:
ds_train[0][0].shape == (30, 9)

True

In [9]:
ds_train[0][1].shape == (1,)

True